# 05 — Gerar com o modelo treinado do zero

Use este caderno depois do `01_modelo_puro_do_zero.ipynb` para testar prompts sem repetir o treinamento. O modelo didático é pequeno: avalie se ele reproduz padrões do corpus, não conhecimento geral.


In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = Path('artifacts/modelo_puro')
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('Modelo ausente. Execute primeiro o caderno 01 até a célula de salvamento.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(device).eval()
print(f'Modelo carregado em {device}; contexto: {model.config.n_positions} tokens')


## Testar prompts

Mude `prompt` e compare amostragem com geração determinística (`do_sample=False`).

In [ ]:
prompt = 'Modelos de linguagem'
inputs = tokenizer(prompt, return_tensors='pt').to(device)
limite = model.config.n_positions - inputs['input_ids'].shape[-1]
if limite <= 0:
    raise ValueError('O prompt ocupa todo o contexto; use um prompt menor.')

with torch.no_grad():
    saida = model.generate(
        **inputs,
        max_new_tokens=min(80, limite),
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(saida[0], skip_special_tokens=True))


## Exercício

Crie cinco prompts de continuação retirados do corpus, anote quais respostas são coerentes e altere apenas uma variável de geração por vez.